In [1]:
!pip install -q -U langchain langchain-core langchain-community langchain-groq langchain-text-splitters faiss-cpu sentence-transformers pypdf groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 10.9 MB/s eta 0:00:00


In [3]:
import os
import urllib.request
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

os.environ["GROQ_API_KEY"] = "enter-your-key"

In [4]:
PDF_PATH = "/content/attention.pdf"
urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", PDF_PATH)
print("PDF downloaded.")
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Total pages: {len(pages)}")

PDF downloaded.
Total pages: 15


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"Total chunks: {len(chunks)}")

Total chunks: 93


In [6]:
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("FAISS index ready!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index ready!


In [7]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded.


In [8]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)
print("LLM connected.")

LLM connected.


In [10]:
prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about a research paper.
Use the context below to answer the question. Be concise and direct.

Context:
{context}

Question: {question}
Answer:
""")

retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

def rerank(question, docs, top_k=4):
    pairs = [(question, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), reverse=True)
    return [doc for _, doc in ranked[:top_k]]

def retrieve_and_rerank(question):
    docs = retriever.invoke(question)
    reranked = rerank(question, docs)
    return "\n\n".join(doc.page_content for doc in reranked)

qa_chain = (
    {"context": RunnablePassthrough() | retrieve_and_rerank, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)
print("RAG pipeline with reranking ready!")

RAG pipeline with reranking ready!


In [13]:
questions = [
    "What problem does the Transformer architecture solve?",
    "What is multi-head attention?",
    "What datasets were used for training?"
]

for question in questions:
    # Show reranking scores
    docs = retriever.invoke(question)
    pairs = [(question, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), reverse=True)
    print(f"\nTop reranked chunk score: {ranked[0][0]:.4f}")

    result = qa_chain.invoke(question)
    print("=" * 60)
    print(f"QUESTION: {question}")
    print(f"ANSWER: {result}")


Top reranked chunk score: 1.3546
QUESTION: What problem does the Transformer architecture solve?
ANSWER: The Transformer architecture solves the problem of relying on recurrent layers in encoder-decoder architectures, replacing them with multi-headed self-attention to improve training speed and model dependencies without regard to distance in input or output sequences.

Top reranked chunk score: 6.7546
QUESTION: What is multi-head attention?
ANSWER: Multi-head attention is a mechanism that allows the model to jointly attend to information from different representation subspaces at different positions. It consists of multiple attention layers running in parallel, each with its own set of weights and projections.

Top reranked chunk score: 3.1632
QUESTION: What datasets were used for training?
ANSWER: The datasets used for training are:

1. WMT 2014 English-German dataset (4.5 million sentence pairs)
2. WMT 2014 English-French dataset (36 million sentences)
3. Wall Street Journal (WSJ) 